# Datamine UNIFCOND Process Example

This notebook demonstrates how to configure and run the **`unifcond`** process wrapper in `dmstudio`.

## Process Description

## UNIFCOND

UNIFCOND performs uniform conditioning (UC).

UC is a non-linear estimation technique which is used to determine the distribution of SMU grades above specified values (cut-off-grades), inside a panel. The principle of the tool is that data exists to accurately estimate grade at a larger resource-level panel scale - but - you cannot accurately estimate the selective mining unit (SMU). This is more likely to be the case where sample data is limited, which is often the case at the start of operations such as during the exploration phase of a project. Linear estimation at this stage will commonly have the effect of smoothing estimates and displaying an apparent loss of ore and metal tonnage at high cut-off grades.

It is implemented interactively via the [Uniform Conditioning Wizard](<../Uniform_Conditioning/UniformConditioning_Introduction.md>).

### Input Files:

* **samples** (Undefined):
  A Datamine file that contains sample positional information and supporting attributes.
  Required=Yes

* **vgram** (Undefined):
  A Datamine experimental [variogram parameter
  file](<../COMMON/filetype.md#VariogramExp>)
  Required=Yes

* **infoeff** (Undefined):
  You can (optionally) incorporate the information effect to the estimation of the grade
  tonnage curves: during the production stage, the actual grades are recovered and may
  then be taken into account so the decision between ore and waste is made upon more
  accurate estimates of the SMUs. Therefore you can anticipate future decisions before
  obtaining the production blast-holes results, because only the kriging variance of these
  SMUs final estimates is necessary.
  Required=No

* **panmodel** (Undefined):
  A file containing the panel model to be conditioned.
  Required=Yes

* **cutoff** (Undefined):
  A file specifying variably-spaced cut-off values. The file should contain a field
  titled **CUTOFF** , which is a list of values to use instead of those in **CUTMIN** ,
  **CUTINT** and **CUTNUM**. If @**CUTOFF** is specified, **CUTMIN** , **CUTINT** and
  **CUTNUM** parameters are ignored.
  Required=No

### Output Files:

* **ucmodel** (Undefined):
  The output file containing the uniform conditioned panel model.
  Required=Yes

### Fields:

* **grade** (Alphanumeric : IN):
  The grade field (present in the samples file) that will be considered during the
  process of Uniform Conditioning.
  Default=Undefined
  Required=Yes

* **weight** (Alphanumeric : IN):
  An optional weighting field.
  Default=Undefined
  Required=No

* **kriging** (Alphanumeric : IN):
  The field in the input (panel) model containing kriged values to be conditioned.
  Default=Undefined
  Required=Yes

* **dispvar** (Alphanumeric : IN):
  The field in the input (panel) model containing kriging variance data.
  Default=Undefined
  Required=Yes

* **errcode** (Alphanumeric : IN):
  The field in the input model containing [error code
  information](<../Uniform_Conditioning/UC%20Error%20Codes.md>), if present.
  Default=Undefined
  Required=No

### Parameters:

* **vrefnum**:
  A reference number relating to an experimental variogram as defined in
  **[VGRAM](<vgram.md>)**.
  Range=
  Values=
  Default=0
  Required=No

* **discx**:
  Number of discretisation points in X direction as used for calculating the covariance
  of a cell with each of the surrounding samples. This is then used in calculating the
  kriging weights.
  Range=
  Values=
  Default=5
  Required=No

* **discy**:
  Number of discretisation points in Y direction
  Range=
  Values=
  Default=5
  Required=No

* **discz**:
  Number of discretisation points in Z direction
  Range=
  Values=
  Default=5
  Required=No

* **normsill**:
  If @**NORMSILL** =1 then the input variogram needs to be normalized by the process If
  @**NORMSILL** =0 then the input variogram model does not need to be normalized by the
  process
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **classes**:
  Number of panel classes
  Range=
  Values=
  Default=1
  Required=No

* **cutmin**:
  The minimum cutoff grade to be considered during Uniform Conditioning
  Range=
  Values=
  Default=0
  Required=No

* **cutint**:
  The size of each grade cutoff interval
  Range=
  Values=
  Default=10
  Required=No

* **cutnum**:
  The number of grade cutoff intervals to considered during Uniform Conditioning
  Range=
  Values=
  Default=10
  Required=No

* **numsmux**:
  Number of selective mining units (per panel) in the X direction
  Range=
  Values=
  Default=1
  Required=No

* **numsmuy**:
  Number of selective mining units (per panel) in the Y direction
  Range=
  Values=
  Default=1
  Required=No

* **numsmuz**:
  Number of selective mining units (per panel) in the Z direction
  Range=
  Values=
  Default=1
  Required=No

* **gaout**:
  Set to 1 to include grade-above cutoffs in the output model. Set 0 to exclude
  grade-above cutoffs which allows 50% more cutoff intervals to be specified.
  Range=0,1
  Values=0,1
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('unifcond')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute unifcond
print("Running unifcond...")
dm_cmd.unifcond(
    samples_i=['t_assays'],  # required
    vgram_i='t_input_file',  # required
    panmodel_i='t_mod1',  # required
    cutoff_i='optional',  # required
    ucmodel_o='t_unifcond_out',  # required
    grade_f='optional',  # required
    weight_f='optional',  # required
    kriging_f='optional',  # required
    dispvar_f='optional',  # required
    # infoeff_i='optional',  # optional
    # errcode_f='optional',  # optional
    # vrefnum_p=0,  # optional
    # discx_p=5,  # optional
    # discy_p=5,  # optional
    # discz_p=5,  # optional
    # normsill_p=0,  # optional
    # classes_p=1,  # optional
    # cutmin_p=0,  # optional
    # cutint_p=10,  # optional
    # cutnum_p=10,  # optional
    # numsmux_p=1,  # optional
    # numsmuy_p=1,  # optional
    # numsmuz_p=1,  # optional
    # gaout_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("unifcond execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_unifcond_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")